# 01f — Data Dictionary, Provenance & Modeling Handoff (Build 4, `v1`)

The final Person 1 deliverable. Produces the **data dictionary** and the **freeze manifest** that let
the modeling owner (Person 2) start the primary survival analysis **without reopening a single raw
source table**.

## This build documents. It does not decide.
No model, no transformation, no imputation, no exclusion, no cognitive selection, no risk groups, no
v2 cohort. The eight pre-anchor-dementia participants **remain in frozen v1**. The nine open scientific
decisions are recorded as **UNRESOLVED**, not answered.

Builds 1–3 are hashed before and after and asserted **byte-identical**.

## 0. Setup and read-only guards

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "Data" / "raw"
FREEZE_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"
NB_DIR = PROJECT_ROOT / "notebooks"
REPORTS = PROJECT_ROOT / "reports"

VERSION = "v1"
MANIFEST_SCHEMA_VERSION = "1.0.0"
RANDOM_SEED = 20260711
ALIGN_TOL_DAYS = 90
DAYS_PER_MONTH = 30.4375

BUILD1 = ["mci_survival_anchor_cohort_v1.tsv", "mci_survival_followup_cohort_v1.tsv",
          "mci_survival_primary_cohort_v1.tsv", "mci_survival_exclusion_log_v1.tsv",
          "mci_survival_cohort_flow_v1.tsv", "mci_survival_freeze_provenance_v1.json"]
BUILD2 = ["mci_survival_manual_qc_cases_v1.tsv", "mci_survival_manual_qc_longitudinal_context_v1.tsv",
          "mci_survival_manual_qc_form_v1.tsv", "mci_survival_manual_qc_sampling_summary_v1.tsv"]
BUILD3 = ["mci_survival_feature_missingness_v1.tsv", "mci_survival_model_complete_case_counts_v1.tsv",
          "mci_survival_feature_timing_v1.tsv", "mci_survival_feature_scenario_counts_v1.tsv",
          "mci_survival_ptau181_platform_availability_v1.tsv"]
PROTECTED = BUILD1 + BUILD2 + BUILD3
BUILD4 = [f"mci_survival_data_dictionary_{VERSION}.tsv", f"mci_survival_freeze_manifest_{VERSION}.json"]

# ---- THE authoritative modeling contract (verified against the file, never assumed) ----
AUTHORITATIVE_FILE = f"mci_survival_primary_cohort_{VERSION}.tsv"
DURATION_COL = "time_to_event_or_censor_days"
EVENT_COL = "event_indicator"
MODEL0_PREDICTORS = ["entry_age", "APOE4_COUNT"]
MODEL1_PREDICTORS = ["entry_age", "APOE4_COUNT", "ptau217"]

ASSERTIONS: list[dict] = []


def check(name, ok, detail=""):
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": str(detail)})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def nb_source_sha256(path: Path) -> str:
    """Hash the notebook's CODE, not its outputs -- stable across executions.
    This is how the self-referential hashing problem is solved (see manifest notes)."""
    doc = json.loads(path.read_text())
    src = "\n".join("".join(c.get("source", [])) for c in doc.get("cells", []))
    return hashlib.sha256(src.encode("utf-8")).hexdigest()


def save(df, name):
    assert name not in PROTECTED, f"REFUSING to overwrite a protected Build 1-3 artifact: {name}"
    p = FREEZE_DIR / name
    df.to_csv(p, sep="\t", index=False)
    print(f"  saved -> {p.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return p


HASHES_BEFORE = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
print(f"Hashed {len(PROTECTED)} protected Build 1-3 artifacts (read-only).")

Hashed 15 protected Build 1-3 artifacts (read-only).


## 1. Load the frozen package and **verify the modeling contract against the file**

The authoritative column names are read **from the file**, never assumed.

In [2]:
PRIM = pd.read_csv(FREEZE_DIR / AUTHORITATIVE_FILE, sep="\t")
SURV = pd.read_csv(FREEZE_DIR / f"mci_survival_followup_cohort_{VERSION}.tsv", sep="\t")
ANCH = pd.read_csv(FREEZE_DIR / f"mci_survival_anchor_cohort_{VERSION}.tsv", sep="\t")
FLOW = pd.read_csv(FREEZE_DIR / f"mci_survival_cohort_flow_{VERSION}.tsv", sep="\t")
QCFORM = pd.read_csv(FREEZE_DIR / f"mci_survival_manual_qc_form_{VERSION}.tsv", sep="\t")
SCEN = pd.read_csv(FREEZE_DIR / f"mci_survival_feature_scenario_counts_{VERSION}.tsv", sep="\t")
MODELS = pd.read_csv(FREEZE_DIR / f"mci_survival_model_complete_case_counts_{VERSION}.tsv", sep="\t")
P181 = pd.read_csv(FREEZE_DIR / f"mci_survival_ptau181_platform_availability_{VERSION}.tsv", sep="\t")
prov1 = json.loads((FREEZE_DIR / f"mci_survival_freeze_provenance_{VERSION}.json").read_text())

PRIM_COLS = list(PRIM.columns)
N_PRIM, EV_PRIM = len(PRIM), int((PRIM[EVENT_COL] == 1).sum())
N_SURV, EV_SURV = len(SURV), int((SURV["event_indicator"] == 1).sum())
N_ANCHOR = len(ANCH)

# ---- the contract must EXIST in the file ----
for c in [DURATION_COL, EVENT_COL] + MODEL1_PREDICTORS:
    check(f"[contract] column '{c}' exists in {AUTHORITATIVE_FILE}", c in PRIM_COLS)

# ---- ASSERTIONS 1, 2, 3, 6, 7 ----
check("primary cohort contains 401 UNIQUE participants",
      N_PRIM == 401 and PRIM["RID"].nunique() == 401, f"N={N_PRIM}, unique={PRIM['RID'].nunique()}")
check("primary cohort contains 85 events", EV_PRIM == 85, f"events={EV_PRIM}")
check("no participant ID is duplicated", not PRIM["RID"].duplicated().any())
check("all follow-up durations are strictly positive",
      bool((PRIM[DURATION_COL] > 0).all()), f"min={PRIM[DURATION_COL].min()}")
check("event column contains ONLY the documented coding {0, 1} with no missing",
      set(PRIM[EVENT_COL].dropna().unique()) <= {0, 1} and PRIM[EVENT_COL].notna().all(),
      f"values={sorted(PRIM[EVENT_COL].unique())}")

# ---- ASSERTIONS 4, 5: Model 0 and Model 1 sit on the SAME participants ----
m0 = PRIM[MODEL0_PREDICTORS].notna().all(axis=1)
m1 = PRIM[MODEL1_PREDICTORS].notna().all(axis=1)
check("Model 0 and Model 1 complete-case participant IDs are IDENTICAL "
      "(adding p-tau217 costs no participants -> the comparison is not confounded)",
      set(PRIM.loc[m0, "RID"]) == set(PRIM.loc[m1, "RID"]) == set(PRIM["RID"]),
      f"M0={int(m0.sum())}, M1={int(m1.sum())}, cohort={N_PRIM}")
check("all Model 1 predictors are nonmissing in the primary cohort",
      bool(m1.all()), {c: int(PRIM[c].isna().sum()) for c in MODEL1_PREDICTORS}.__str__())

print(f"\nFrozen: A={N_ANCHOR}  B={N_SURV} ({EV_SURV} ev)  C={N_PRIM} ({EV_PRIM} ev)")
print(f"Authoritative file : {AUTHORITATIVE_FILE}  ({N_PRIM} x {len(PRIM_COLS)})")
print(f"duration={DURATION_COL} | event={EVENT_COL}")
print(f"Model 0 = {' + '.join(MODEL0_PREDICTORS)}")
print(f"Model 1 = {' + '.join(MODEL1_PREDICTORS)}")

  PASS  [contract] column 'time_to_event_or_censor_days' exists in mci_survival_primary_cohort_v1.tsv
  PASS  [contract] column 'event_indicator' exists in mci_survival_primary_cohort_v1.tsv
  PASS  [contract] column 'entry_age' exists in mci_survival_primary_cohort_v1.tsv
  PASS  [contract] column 'APOE4_COUNT' exists in mci_survival_primary_cohort_v1.tsv
  PASS  [contract] column 'ptau217' exists in mci_survival_primary_cohort_v1.tsv
  PASS  primary cohort contains 401 UNIQUE participants  [N=401, unique=401]
  PASS  primary cohort contains 85 events  [events=85]
  PASS  no participant ID is duplicated
  PASS  all follow-up durations are strictly positive  [min=1.0]
  PASS  event column contains ONLY the documented coding {0, 1} with no missing  [values=[np.int64(0), np.int64(1)]]
  PASS  Model 0 and Model 1 complete-case participant IDs are IDENTICAL (adding p-tau217 costs no participants -> the comparison is not confounded)  [M0=401, M1=401, cohort=401]
  PASS  all Model 1 predicto

## 2. Recompute the Build 2 / Build 3 headline facts **from the artifacts**

Nothing below is hard-coded; every number is derived from the frozen files so the handoff cannot
drift from reality.

In [3]:
PRE_DEM = ANCH[ANCH["qc_dementia_on_or_before_anchor"].astype(bool)]
pd_anchor = len(PRE_DEM)
pd_surv = int(PRE_DEM["followup_eligible"].astype(bool).sum())
pd_prim = int(PRE_DEM["primary_eligible"].astype(bool).sum())
pd_surv_ev = int(((PRE_DEM["followup_eligible"].astype(bool)) & (PRE_DEM["event_indicator"] == 1)).sum())
pd_prim_ev_rids = sorted(PRE_DEM.loc[(PRE_DEM["primary_eligible"].astype(bool))
                                     & (PRE_DEM["event_indicator"] == 1), "RID"].astype(int))
pd_prim_ev = len(pd_prim_ev_rids)

# short follow-up
cia = SURV[(SURV["event_indicator"] == 0)
           & SURV["censor_src_row"].notna()
           & (SURV["censor_src_row"] == SURV["anchor_dx_src_row"])]
n_cia = len(cia)
n_fu30 = int((SURV["time_to_event_or_censor_days"] <= 30).sum())
n_fu90 = int((SURV["time_to_event_or_censor_days"] <= 90).sum())

# prior-dementia exclusion scenario, read from the Build 3 scenario table (NOT hard-coded)
s2 = SCEN[SCEN["scenario_id"] == "S2_exclude_prior_dementia"].iloc[0]
s2_prim, s2_prim_ev = int(s2["primary_n"]), int(s2["primary_events"])

# secondary feasibility, read from the Build 3 model table
def mrow(mid, rule=None):
    d = MODELS[MODELS["model_id"] == mid]
    if rule is not None:
        d = d[d["timing_rule"] == rule]
    r = d.iloc[0]
    return int(r["complete_case_n"]), int(r["event_n"])


m_gfap, e_gfap = mrow("M5")
m_ratio, e_ratio = mrow("M6")
p181_qx = P181[P181["platform_column"] == "ptau181_qx"].iloc[0]
p181_ro = P181[P181["platform_column"] == "ptau181_roche"].iloc[0]

print(f"PRE-ANCHOR DEMENTIA (recomputed): anchor={pd_anchor}  survival={pd_surv} (events {pd_surv_ev})"
      f"  primary={pd_prim} (events {pd_prim_ev} -> RID {pd_prim_ev_rids})")
print(f"  exclusion scenario -> primary {s2_prim} / {s2_prim_ev} events")
print(f"SHORT FOLLOW-UP (recomputed): censor-is-anchor-visit={n_cia}  fu<=30d={n_fu30}  fu<=90d={n_fu90}")
print(f"SECONDARY: +GFAP {m_gfap}/{e_gfap} events | +Ab42/Ab40 {m_ratio}/{e_ratio} events")
print(f"p-tau181 anchor-aligned: QX={int(p181_qx['anchor_aligned_pm90_primary'])}, "
      f"Roche={int(p181_ro['anchor_aligned_pm90_primary'])} (primary cohort)")

# ---- ASSERTION 15 (part): no human QC decision has been populated ----
HUMAN = [c for c in QCFORM.columns if c.startswith(("qc_", "reviewer_", "review_"))
         or c == "recommended_action"]
check("no human QC decision has been silently populated (all review fields still blank)",
      bool(QCFORM[HUMAN].isna().all().all()), f"{len(HUMAN)} review fields x {len(QCFORM)} rows")
check("frozen v1 STILL contains all pre-anchor-dementia participants (none excluded by Build 4)",
      pd_anchor == 8 and pd_prim == 4, f"anchor={pd_anchor}, primary={pd_prim}")

PRE-ANCHOR DEMENTIA (recomputed): anchor=8  survival=5 (events 2)  primary=4 (events 1 -> RID [467])
  exclusion scenario -> primary 397 / 84 events
SHORT FOLLOW-UP (recomputed): censor-is-anchor-visit=74  fu<=30d=62  fu<=90d=83
SECONDARY: +GFAP 361/85 events | +Ab42/Ab40 399/83 events
p-tau181 anchor-aligned: QX=1, Roche=3 (primary cohort)
  PASS  no human QC decision has been silently populated (all review fields still blank)  [13 review fields x 46 rows]
  PASS  frozen v1 STILL contains all pre-anchor-dementia participants (none excluded by Build 4)  [anchor=8, primary=4]


## 3. Data dictionary — one row for **every** column of the authoritative file

78 columns, in file order. `cohort_role` tells Person 2 exactly what each column is for; only columns
marked `authoritative_for_modeling = TRUE` may enter a model.

In [4]:
def D(col, label, unit, allowed, ds, field, deriv, timing, missing, sentinel, platform,
      raw_der, role, transform, handling, caveat, auth):
    return dict(output_column=col, label=label, unit=unit, allowed_values=allowed,
                source_dataset=ds, source_field=field, derivation=deriv, timing_rule=timing,
                missing_value_rule=missing, sentinel_handling=sentinel, assay_platform=platform,
                raw_or_derived=raw_der, cohort_role=role, transformation_allowed=transform,
                recommended_downstream_handling=handling, known_caveat=caveat,
                authoritative_for_modeling=auth)


PLASMA = "UPENN_PLASMA_FUJIREBIO_QUANTERIX"
DXSUM = "DXSUM"
NA = "n/a"
SENT_PLASMA = "-4/-5 -> NaN; non-positive -> NaN (existing rule)"
SENT_NONE = "none applied"
SRCROW = ("0-based positional index of the record in the raw CSV as delivered, stamped immediately "
          "after read_csv and BEFORE any filtering/sorting/dedup")
NEVER = "NO — never enter a model"

DD = [
 # ---------- identification ----------
 D("RID", "Participant ID (ADNI roster ID)", "integer id", "positive integer", "DXSUM/plasma", "RID",
   "join key across all tables", "time-invariant", "never missing", NA, "", "raw", "identifier",
   NEVER, "use as the participant key; one row per RID", "", "FALSE"),
 D("PTID", "Participant ID (human-readable)", "string", "e.g. 011_S_0008", "DXSUM", "PTID",
   "carried for readability", "time-invariant", "never missing", NA, "", "raw", "identifier",
   NEVER, "display only", "", "FALSE"),
 # ---------- anchor ----------
 D("anchor_date", "Plasma anchor date (survival time origin)", "date YYYY-MM-DD", "valid date",
   PLASMA, "EXAMDATE", "earliest eligible plasma draw (>=1 usable core assay) whose nearest "
   "diagnosis within +/-90d is MCI", "THE baseline; t = 0", "never missing", NA, "", "raw",
   "provenance", NEVER, "use only to recompute offsets; the duration column is already derived",
   "later or more complete draws are NEVER preferred", "FALSE"),
 D("anchor_viscode", "Visit code of the anchor plasma draw", "string", "ADNI VISCODE2", PLASMA,
   "VISCODE2", "carried from the anchor row", "at anchor", "may be missing", NA, "", "raw",
   "provenance", NEVER, "QC only", "", "FALSE"),
 D("anchor_src_row", "Source row of the selected plasma draw", "integer index", ">=0", PLASMA,
   "(generated)", SRCROW, "at anchor", "never missing", NA, "", "derived", "provenance", NEVER,
   "use to trace the anchor back to the raw plasma CSV",
   "the UPENN plasma panel has NO native row id — this generated index is the only one", "FALSE"),
 D("anchor_dx_date", "Date of the MCI diagnosis matched to the anchor", "date YYYY-MM-DD",
   "valid date", DXSUM, "EXAMDATE", "nearest MCI diagnosis within +/-90d of the anchor",
   "within +/-90d of anchor", "never missing", NA, "", "raw", "provenance", NEVER, "QC only",
   "may fall AFTER the anchor (negative align_offset_days)", "FALSE"),
 D("anchor_dx_viscode", "Visit code of the matched MCI diagnosis", "string", "ADNI VISCODE2", DXSUM,
   "VISCODE2", "carried from the matched row", "within +/-90d", "may be missing", NA, "", "raw",
   "provenance", NEVER, "QC only", "", "FALSE"),
 D("anchor_dx_src_row", "Source row of the matched MCI diagnosis", "integer index", ">=0", DXSUM,
   "(generated)", SRCROW, "within +/-90d", "never missing", NA, "", "derived", "provenance", NEVER,
   "trace the matched MCI diagnosis to the raw DXSUM CSV", "", "FALSE"),
 D("anchor_dx_src_id", "Native ADNI row id of the matched MCI diagnosis", "integer", "ADNI ID",
   DXSUM, "ID", "carried from DXSUM", "within +/-90d", "never missing", NA, "", "raw", "provenance",
   NEVER, "alternative trace key", "", "FALSE"),
 D("align_offset_days", "Signed plasma-to-diagnosis gap (anchor - matched dx)", "days",
   "-90 .. +90", "derived", "anchor_date - anchor_dx_date", "(anchor - dx).days", "at anchor",
   "never missing", NA, "", "derived", "QC flag", "NO",
   "review large |offset| in QC; negative = the MCI diagnosis came AFTER the draw", "", "FALSE"),
 D("align_offset_days_abs", "Absolute plasma-to-diagnosis gap", "days", "0 .. 90", "derived",
   "abs(align_offset_days)", "abs()", "at anchor", "never missing", NA, "", "derived", "QC flag",
   "NO", "26 participants sit in the 70-90d band", "", "FALSE"),
 D("same_day_alignment", "Plasma draw and MCI diagnosis on the same day", "boolean", "TRUE/FALSE",
   "derived", "align_offset_days == 0", "abs(offset)==0", "at anchor", "never missing", NA, "",
   "derived", "QC flag", NEVER, "QC only", "", "FALSE"),
 D("anchor_match_type", "Same-day vs cross-visit anchor match", "string",
   "same_day | cross_visit", "derived", "same_day_alignment", "label of the flag", "at anchor",
   "never missing", NA, "", "derived", "QC flag", NEVER, "QC only", "", "FALSE"),
 D("baseline_dx_harmonized", "Diagnosis at baseline", "string", "MCI (constant)", DXSUM,
   "DIAGNOSIS", "MCI by construction — the anchor requires an MCI match", "at anchor",
   "never missing", NA, "", "derived", "provenance", NEVER,
   "constant; confirms every participant is MCI at baseline", "", "FALSE"),
 D("n_usable_core_assays", "Count of usable core plasma assays at the anchor draw", "count",
   "1 .. 5", PLASMA, "pT217_F/AB42_F/AB40_F/NfL_Q/GFAP_Q", "count of non-sentinel, positive core "
   "assays", "at anchor", "never missing", SENT_PLASMA, "Fujirebio + Quanterix", "derived",
   "QC flag", "NO", "the broad anchor requires >=1", "", "FALSE"),
 D("anchor_tiebreak_rule", "Documented anchor tie-break", "string", "constant text", "derived",
   "(constant)", "records the deterministic tie-break used", NA, "never missing", NA, "", "derived",
   "provenance", NEVER, "documentation only", "", "FALSE"),
 # ---------- PRIMARY: age ----------
 D("entry_age", "Age at study entry — THE authoritative age variable", "years",
   "positive float (~55-95)", "My_Table", "entry_age",
   "joined subject_id(==PTID) -> RID; used AS-IS",
   "UNDATED — no datestamp exists in the source; age-at-anchor is NOT validly reconstructable",
   "no imputation; 0 missing in the survival cohort", "none in source", "",
   "raw", "PRIMARY PREDICTOR",
   "YES — centering/scaling/splines are modeling-stage decisions",
   "use as the age term in Model 0 and Model 1, exactly as provided",
   "UNDATED. It is age at STUDY ENTRY, not at the anchor; the anchor can be years later "
   "(median 0.14y, p90 7.1y). No source date or timing offset was fabricated. Do NOT substitute "
   "age_at_anchor_approx.", "TRUE"),
 D("age_src_row", "Source row of the entry_age record", "integer index", ">=0", "My_Table",
   "(generated)", SRCROW, NA, "never missing", NA, "", "derived", "provenance", NEVER,
   "trace to the raw My_Table CSV", "", "FALSE"),
 D("age_source_date", "Date of the age measurement", "date", "ALWAYS EMPTY", "My_Table", "(none)",
   "structurally absent — My_Table carries no date column", "NONE",
   "always missing BY CONSTRUCTION", NA, "", "derived", "provenance", NEVER,
   "do not populate; the absence is the finding",
   "deliberately empty. NO date was fabricated.", "FALSE"),
 D("entry_age_has_datestamp", "Does entry_age carry a datestamp?", "boolean", "FALSE (constant)",
   "derived", "(constant)", "records the limitation explicitly", NA, "never missing", NA, "",
   "derived", "provenance", NEVER, "documentation only",
   "always FALSE — this is the age limitation in one column", "FALSE"),
 D("study_entry_date_proxy", "PROXY study-entry date (earliest dated DXSUM visit)", "date",
   "valid date", DXSUM, "EXAMDATE (earliest)", "earliest dated diagnosis visit per participant",
   "proxy only", "may be missing", NA, "", "derived", "provenance", NEVER,
   "QC only — NOT an age date",
   "an ASSUMPTION, not documented in the source. Never treat as the entry_age date.", "FALSE"),
 D("years_entry_to_anchor", "Years from the entry proxy to the anchor", "years", "float",
   "derived", "(anchor - study_entry_date_proxy)/365.25", "gap in years", "proxy-based",
   "may be missing", NA, "", "derived", "provenance", NEVER,
   "use ONLY to judge how stale entry_age is for a participant",
   "28 participants have a NEGATIVE value (anchor precedes the entry proxy) -> "
   "age_derivation_suspect", "FALSE"),
 D("age_at_anchor_approx", "APPROXIMATE age at anchor — NOT AUTHORITATIVE", "years", "float",
   "derived", "entry_age + years_entry_to_anchor", "entry_age plus the proxy gap", "anchor (approx)",
   "may be missing", NA, "", "derived", "provenance", NEVER,
   "DO NOT USE AS THE MODEL'S AGE VARIABLE",
   "rests on an undocumented assumption (entry_age is measured at the earliest DXSUM visit). "
   "Provided for transparency only. Use entry_age.", "FALSE"),
 D("age_is_entry_age_fallback", "Flag: age is entry_age, not age-at-anchor", "boolean",
   "TRUE (constant)", "derived", "(constant)", "records the fallback", NA, "never missing", NA, "",
   "derived", "QC flag", NEVER, "documentation only", "always TRUE", "FALSE"),
 D("age_derivation_suspect", "Flag: anchor precedes the study-entry proxy", "boolean", "TRUE/FALSE",
   "derived", "years_entry_to_anchor < 0", "negative gap", NA, "never missing", NA, "", "derived",
   "QC flag", NEVER, "NOT an exclusion; review if age is ever re-derived", "", "FALSE"),
 # ---------- PRIMARY: APOE ----------
 D("GENOTYPE", "APOE genotype string", "string", "e.g. 2/3, 3/3, 3/4, 4/4", "APOERES", "GENOTYPE",
   "raw genotype", "time-invariant (germline)", "missing for 9 survival-cohort participants",
   SENT_NONE, "", "raw", "provenance", NEVER, "source of APOE4_COUNT; keep for audit", "", "FALSE"),
 D("APOE4_COUNT", "Number of APOE e4 alleles", "alleles", "0, 1, 2",
   "APOERES", "GENOTYPE", "count of the character '4' in GENOTYPE", "time-invariant (germline)",
   "NO IMPUTATION. Missing for 9 of the 410 survival participants — those 9 are EXCLUDED from the "
   "primary cohort (this is the entire 410->401 drop; 1 of them is an event)",
   SENT_NONE, "", "derived", "PRIMARY PREDICTOR",
   "YES — treat as linear count or as a factor; that is a modeling-stage decision",
   "use in Model 0 and Model 1. Nonmissing for all 401 primary participants.",
   "0 alleles in 215 primary participants — a legitimate value, not missing", "TRUE"),
 D("apoe_src_row", "Source row of the APOE record", "integer index", ">=0", "APOERES", "(generated)",
   SRCROW, NA, "missing where GENOTYPE is missing", NA, "", "derived", "provenance", NEVER,
   "trace to the raw APOERES CSV", "", "FALSE"),
 D("apoe_src_id", "Native ADNI row id of the APOE record", "integer", "ADNI ID", "APOERES", "ID",
   "carried from APOERES", NA, "missing where GENOTYPE is missing", NA, "", "raw", "provenance",
   NEVER, "alternative trace key", "", "FALSE"),
 # ---------- PRIMARY: p-tau217 ----------
 D("ptau217", "Plasma p-tau217 — THE primary biomarker", "pg/mL", "positive float",
   PLASMA, "pT217_F", "raw assay value at the anchor draw; sentinels and non-positive values -> NaN",
   "measured AT the anchor draw (offset = 0 by construction)",
   "ZERO missingness in the survival cohort after the existing validity rules (0 sentinels existed)",
   SENT_PLASMA, "Fujirebio (Lumipulse)", "raw", "PRIMARY PREDICTOR",
   "YES — log-transform / scaling are MODELING-STAGE decisions, deliberately NOT applied here",
   "the added term in Model 1. Distribution is right-skewed; a log transform is likely but is "
   "Person 2's call.",
   "NO transformation of any kind has been applied. Values are raw pg/mL.", "TRUE"),
 D("ptau217_assay", "p-tau217 assay/source identifier", "string", "constant text", PLASMA,
   "(constant)", "records the assay provenance", "at anchor", "never missing", NA,
   "Fujirebio", "derived", "provenance", NEVER, "documentation only", "", "FALSE"),
 # ---------- SECONDARY plasma ----------
 D("gfap", "Plasma GFAP", "pg/mL", "positive float", PLASMA, "GFAP_Q",
   "raw assay at the anchor draw", "at anchor (offset 0)",
   "missing for 43 of 410 (ALL from -4/-5 sentinel removal)", SENT_PLASMA, "Quanterix (Simoa)",
   "raw", "SECONDARY PREDICTOR", "YES (modeling-stage)",
   "the CHEAPEST secondary addition: 361 complete cases retaining ALL 85 events",
   "adding it loses 40 participants but ZERO events", "FALSE"),
 D("nfl", "Plasma NfL", "pg/mL", "positive float", PLASMA, "NfL_Q", "raw assay at the anchor draw",
   "at anchor (offset 0)", "missing for 43 of 410 (sentinel removal)", SENT_PLASMA,
   "Quanterix (Simoa)", "raw", "SECONDARY PREDICTOR", "YES (modeling-stage)",
   "not in the pre-specified secondary set; same coverage as GFAP", "", "FALSE"),
 D("abeta42", "Plasma Abeta-42", "pg/mL", "positive float", PLASMA, "AB42_F",
   "raw assay at the anchor draw", "at anchor (offset 0)", "missing for 2 of 410 (sentinel removal)",
   SENT_PLASMA, "Fujirebio", "raw", "SECONDARY PREDICTOR", "YES (modeling-stage)",
   "ratio NUMERATOR — must be preserved alongside the ratio", "", "FALSE"),
 D("abeta40", "Plasma Abeta-40", "pg/mL", "positive float", PLASMA, "AB40_F",
   "raw assay at the anchor draw", "at anchor (offset 0)", "missing for 2 of 410 (sentinel removal)",
   SENT_PLASMA, "Fujirebio", "raw", "SECONDARY PREDICTOR", "YES (modeling-stage)",
   "ratio DENOMINATOR — must be preserved alongside the ratio",
   "0 zero and 0 non-positive denominators in the cohort", "FALSE"),
 D("ratio_ab42_ab40", "Abeta-42/Abeta-40 ratio (recomputed)", "ratio (dimensionless)",
   "positive float", "derived", "AB42_F / AB40_F",
   "abeta42 / abeta40 — BOTH components come from the SAME raw plasma row (same participant, draw, "
   "date and platform), so a compatibility mismatch is structurally impossible",
   "at anchor (offset 0)",
   "non-null iff both components are valid and the denominator > 0; 408 of 410 valid (84 events)",
   SENT_PLASMA, "Fujirebio (both components)", "derived", "SECONDARY PREDICTOR",
   "YES (modeling-stage)", "complete-case model: 399 participants / 83 events",
   "0 undefined and 0 infinite ratios. Raw components are preserved.", "FALSE"),
 D("ratio_ab42_ab40_vendor", "Abeta-42/Abeta-40 ratio as supplied by the vendor", "ratio",
   "positive float", PLASMA, "AB42_AB40_F", "vendor-computed ratio, kept SEPARATELY", "at anchor",
   "as supplied", SENT_PLASMA, "Fujirebio", "raw", "provenance", "NO",
   "QC cross-check only — do NOT substitute for ratio_ab42_ab40",
   "agrees with the recomputed ratio to 5e-05 (rounding)", "FALSE"),
 # ---------- cognition ----------
 D("CDRSB", "CDR Sum of Boxes (global severity / staging)", "0-18 (sum of boxes)", "0 .. 18",
   "CDR", "CDRSB", "nearest measurement within +/-90d of the anchor (CDVERSION in {1,2})",
   "NEAREST within +/-90d — MAY FALL AFTER THE ANCHOR",
   "no imputation; 89 of 410 missing", "NONE APPLIED (see caveat)", "", "raw",
   "SECONDARY PREDICTOR (candidate)", "YES (modeling-stage)",
   "NOT approved as a primary predictor. Complete-case cost is severe: 314 participants but only "
   "50 of 85 events.",
   "15 values were measured AFTER the anchor (max +83d) -> potential leakage. Restricting to "
   "on-or-before costs 0 events (51->51). -1 'not-done' codes exist in the CDR family and are NOT "
   "covered by the -4/-5 sentinel rule (none reached v1).", "FALSE"),
 D("cdrsb_date", "Date of the aligned CDR visit", "date", "valid date", "CDR", "VISDATE",
   "date of the selected CDR measurement", "within +/-90d", "missing where CDRSB is missing", NA,
   "", "raw", "provenance", NEVER, "use to audit cognitive timing", "", "FALSE"),
 D("cdrsb_offset_days", "CDR date minus anchor date", "days", "-90 .. +90", "derived",
   "cdrsb_date - anchor_date", "signed offset", "relative to anchor",
   "missing where CDRSB is missing", NA, "", "derived", "QC flag", "NO",
   "POSITIVE values = measured AFTER the blood draw (leakage risk)", "", "FALSE"),
 D("MMSCORE", "MMSE total score (global cognition)", "0-30", "0 .. 30", "MMSE", "MMSCORE",
   "nearest measurement within +/-90d of the anchor", "NEAREST within +/-90d — MAY FALL AFTER",
   "no imputation; 49 of 410 missing", "NONE APPLIED (see caveat)", "", "raw",
   "SECONDARY PREDICTOR (candidate)", "YES (modeling-stage)",
   "NOT approved as a primary predictor. Best event retention of the cognitive options "
   "(354 participants / 81 events).",
   "17 values measured AFTER the anchor (max +83d). Restricting to on-or-before costs 0 events "
   "(82->82). 14 raw '-1' not-done codes exist in MMSE and are NOT covered by the sentinel rule "
   "(none reached v1).", "FALSE"),
 D("mmse_date", "Date of the aligned MMSE visit", "date", "valid date", "MMSE", "VISDATE",
   "date of the selected MMSE", "within +/-90d", "missing where MMSCORE is missing", NA, "", "raw",
   "provenance", NEVER, "use to audit cognitive timing", "", "FALSE"),
 D("mmse_offset_days", "MMSE date minus anchor date", "days", "-90 .. +90", "derived",
   "mmse_date - anchor_date", "signed offset", "relative to anchor",
   "missing where MMSCORE is missing", NA, "", "derived", "QC flag", "NO",
   "POSITIVE = measured AFTER the blood draw", "", "FALSE"),
 D("MOCA", "MoCA total score (global cognition)", "0-30", "0 .. 30", "MOCA", "MOCA",
   "nearest measurement within +/-90d of the anchor", "NEAREST within +/-90d — MAY FALL AFTER",
   "no imputation; 88 of 410 missing", "NONE APPLIED", "", "raw",
   "SECONDARY PREDICTOR (candidate)", "YES (modeling-stage)",
   "NOT approved as a primary predictor. Most expensive cognitive option: 313 participants but "
   "only 41 of 85 events.",
   "24 values measured AFTER the anchor (max +49d).", "FALSE"),
 D("moca_date", "Date of the aligned MoCA visit", "date", "valid date", "MOCA", "VISDATE",
   "date of the selected MoCA", "within +/-90d", "missing where MOCA is missing", NA, "", "raw",
   "provenance", NEVER, "use to audit cognitive timing", "", "FALSE"),
 D("moca_offset_days", "MoCA date minus anchor date", "days", "-90 .. +90", "derived",
   "moca_date - anchor_date", "signed offset", "relative to anchor",
   "missing where MOCA is missing", NA, "", "derived", "QC flag", "NO",
   "POSITIVE = measured AFTER the blood draw", "", "FALSE"),
 D("FAQTOTAL", "FAQ total score (FUNCTIONAL status, not cognition)", "0-30", "0 .. 30", "FAQ",
   "FAQTOTAL", "nearest measurement within +/-90d of the anchor",
   "NEAREST within +/-90d — MAY FALL AFTER", "no imputation; 31 of 410 missing",
   "NONE APPLIED (see caveat)", "", "raw", "SECONDARY PREDICTOR (candidate)",
   "YES (modeling-stage)",
   "NOT approved as a primary predictor. Best coverage (370 participants / 81 events) but the "
   "highest leakage.",
   "51 values measured AFTER the anchor (max +88d) — the highest of any cognitive variable. "
   "32 raw '-1' not-done codes exist in FAQ and are NOT covered by the sentinel rule (none reached "
   "v1). Measures FUNCTION, not cognition.", "FALSE"),
 D("faq_date", "Date of the aligned FAQ visit", "date", "valid date", "FAQ", "VISDATE",
   "date of the selected FAQ", "within +/-90d", "missing where FAQTOTAL is missing", NA, "", "raw",
   "provenance", NEVER, "use to audit cognitive timing", "", "FALSE"),
 D("faq_offset_days", "FAQ date minus anchor date", "days", "-90 .. +90", "derived",
   "faq_date - anchor_date", "signed offset", "relative to anchor",
   "missing where FAQTOTAL is missing", NA, "", "derived", "QC flag", "NO",
   "POSITIVE = measured AFTER the blood draw", "", "FALSE"),
 # ---------- OUTCOME ----------
 D("event_indicator", "SURVIVAL EVENT — dementia observed", "0/1", "1 = dementia, 0 = censored",
   DXSUM, "DIAGNOSIS", "1 if a dementia diagnosis (code 3) occurs STRICTLY AFTER the anchor; "
   "0 if censored", "outcome, strictly post-anchor",
   "never missing in the primary cohort (participants without usable post-anchor follow-up are "
   "EXCLUDED from the survival cohort, never coded 0)", NA, "", "derived", "OUTCOME",
   NEVER, "THE event column. Use as-is: Surv(time_to_event_or_censor_days, event_indicator).",
   "0 does NOT mean 'stable' — it means 'not yet observed to progress'.", "TRUE"),
 D("event_date", "Date of the first post-anchor dementia diagnosis", "date", "valid date or empty",
   DXSUM, "EXAMDATE", "FIRST dementia diagnosis strictly after the anchor",
   "strictly > anchor_date", "empty for censored participants", NA, "", "raw", "provenance",
   NEVER, "provenance only — the duration column is already derived from it",
   "do not recompute durations from dates; use the duration column", "FALSE"),
 D("event_src_row", "Source row of the event diagnosis", "integer index", ">=0 or empty", DXSUM,
   "(generated)", SRCROW, "at the event", "empty for censored", NA, "", "derived", "provenance",
   NEVER, "trace the event to the raw DXSUM CSV", "", "FALSE"),
 D("event_src_id", "Native ADNI row id of the event diagnosis", "integer", "ADNI ID or empty",
   DXSUM, "ID", "carried from DXSUM", "at the event", "empty for censored", NA, "", "raw",
   "provenance", NEVER, "alternative trace key", "", "FALSE"),
 D("censor_date", "Date of the last qualifying post-anchor non-dementia visit", "date",
   "valid date or empty", DXSUM, "EXAMDATE",
   "LAST post-anchor CN/MCI diagnosis when no dementia occurs", "strictly > anchor_date",
   "empty for participants with an event", NA, "", "raw", "provenance", NEVER,
   "provenance only",
   "censoring is ADMINISTRATIVE (last clinical visit) — no death/loss-to-follow-up table exists, "
   "so censoring may be informative", "FALSE"),
 D("censor_src_row", "Source row of the censoring diagnosis", "integer index", ">=0 or empty",
   DXSUM, "(generated)", SRCROW, "at censoring", "empty for events", NA, "", "derived",
   "provenance", NEVER, "trace the censor to the raw DXSUM CSV",
   "when this EQUALS anchor_dx_src_row, the participant's entire follow-up is the anchor-matching "
   "interval (74 such participants; median 12 days)", "FALSE"),
 D("censor_src_id", "Native ADNI row id of the censoring diagnosis", "integer",
   "ADNI ID or empty", DXSUM, "ID", "carried from DXSUM", "at censoring", "empty for events", NA,
   "", "raw", "provenance", NEVER, "alternative trace key", "", "FALSE"),
 D("followup_end_date", "Event-or-censor date", "date", "valid date", "derived",
   "event_date or censor_date", "whichever applies", "strictly > anchor_date", "never missing",
   NA, "", "derived", "provenance", NEVER, "provenance only", "", "FALSE"),
 D("time_to_event_or_censor_days", "SURVIVAL DURATION — days from anchor to event or censor",
   "days", "strictly positive integer-valued float", "derived",
   "followup_end_date - anchor_date",
   "(event_or_censor_date - anchor_date).days; strictly positive by construction",
   "from the anchor (t = 0) to the event or the censor",
   "never missing in the primary cohort", NA, "", "derived", "OUTCOME", NEVER,
   "THE duration column. Use as-is: Surv(time_to_event_or_censor_days, event_indicator).",
   "range 1 - 3694 days (median 425). 62 survival participants have <=30 days — legitimate "
   "censored observations, NOT 'stable'.", "TRUE"),
 D("time_to_event_or_censor_months", "Duration in months (convenience)", "months",
   "positive float", "derived", "time_to_event_or_censor_days / 30.4375",
   "THE ONE documented conversion: days / 30.4375 (= 365.25/12)", "same as the duration column",
   "never missing in the primary cohort", NA, "", "derived", "OUTCOME (convenience)", NEVER,
   "prefer the DAYS column for modeling; this is for reporting",
   "derived from days — do not model both", "FALSE"),
 D("n_post_anchor_visits", "Number of post-anchor diagnosis visits", "count", ">=1", DXSUM,
   "(count)", "count of dated diagnoses strictly after the anchor", "post-anchor", "never missing",
   NA, "", "derived", "QC flag", "NO", "a proxy for follow-up intensity", "", "FALSE"),
 D("censoring_reason", "Why the participant was censored", "string",
   "'last non-dementia (CN/MCI) follow-up' or empty for events", "derived", "(text)",
   "records the censoring rule applied", "at censoring", "empty for events", NA, "", "derived",
   "provenance", NEVER, "documentation only", "", "FALSE"),
 D("reverted_cn", "Participant reverted to CN at some post-anchor visit", "0/1", "0 or 1", DXSUM,
   "DIAGNOSIS", "any post-anchor CN diagnosis", "post-anchor", "never missing", NA, "", "derived",
   "QC flag", "NO", "NOT an exclusion; informational", "", "FALSE"),
 # ---------- inclusion flags ----------
 D("broad_anchor_eligible", "In the broad anchor cohort (A)", "boolean", "TRUE (constant here)",
   "derived", "(flag)", "TRUE for every row of every frozen cohort file", NA, "never missing", NA,
   "", "derived", "QC flag", NEVER, "constant within this file", "", "FALSE"),
 D("followup_eligible", "In the survival-follow-up cohort (B)", "boolean", "TRUE (constant here)",
   "derived", "(flag)", "has >=1 usable post-anchor diagnosis", NA, "never missing", NA, "",
   "derived", "QC flag", NEVER, "constant within the primary file", "", "FALSE"),
 D("primary_eligible", "In the primary complete-case cohort (C)", "boolean", "TRUE (constant here)",
   "derived", "(flag)", "followup_eligible AND entry_age AND APOE4_COUNT AND ptau217",
   NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "constant TRUE within the primary file — it IS the inclusion rule", "", "FALSE"),
 D("final_inclusion_stage", "Deepest cohort reached", "string",
   "primary_complete_case (constant here)", "derived", "(label)", "deepest cohort reached", NA,
   "never missing", NA, "", "derived", "provenance", NEVER, "constant within this file", "",
   "FALSE"),
 D("exclusion_stage", "Stage at which the participant was excluded", "string",
   "ALWAYS EMPTY in the primary file", "derived", "(label)",
   "empty because every row of this file was INCLUDED", NA,
   "always empty here (reads back as NaN)", NA, "", "derived", "provenance", NEVER,
   "see the exclusion log for excluded participants",
   "empty by construction in the primary cohort", "FALSE"),
 D("exclusion_reason", "Why the participant was excluded", "string",
   "ALWAYS EMPTY in the primary file", "derived", "(text)",
   "empty because every row of this file was INCLUDED", NA,
   "always empty here (reads back as NaN)", NA, "", "derived", "provenance", NEVER,
   "see mci_survival_exclusion_log_v1.tsv", "empty by construction", "FALSE"),
 # ---------- availability flags ----------
 D("entry_age_available", "entry_age is nonmissing", "boolean", "TRUE (constant here)", "derived",
   "(flag)", "entry_age.notna()", NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "constant TRUE in the primary file", "", "FALSE"),
 D("apoe4_available", "APOE4_COUNT is nonmissing", "boolean", "TRUE (constant here)", "derived",
   "(flag)", "APOE4_COUNT.notna()", NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "constant TRUE in the primary file", "", "FALSE"),
 D("ptau217_available", "ptau217 is nonmissing", "boolean", "TRUE (constant here)", "derived",
   "(flag)", "ptau217.notna()", NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "constant TRUE in the primary file", "", "FALSE"),
 D("gfap_available", "gfap is nonmissing", "boolean", "TRUE/FALSE", "derived", "(flag)",
   "gfap.notna()", NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "use to build the +GFAP secondary cohort without touching the raw data", "", "FALSE"),
 D("nfl_available", "nfl is nonmissing", "boolean", "TRUE/FALSE", "derived", "(flag)",
   "nfl.notna()", NA, "never missing", NA, "", "derived", "QC flag", NEVER,
   "use to build an +NfL cohort", "", "FALSE"),
 D("amyloid_ratio_available", "Abeta42, Abeta40 and the ratio are all nonmissing", "boolean",
   "TRUE/FALSE", "derived", "(flag)", "all three components notna()", NA, "never missing", NA, "",
   "derived", "QC flag", NEVER, "use to build the +Abeta42/Abeta40 secondary cohort", "", "FALSE"),
 # ---------- QC flags ----------
 D("sameday_dementia_at_anchor", "Dementia diagnosis ON the anchor date", "count", "0 (all rows)",
   DXSUM, "DIAGNOSIS", "count of dementia diagnoses on the anchor date", "at anchor",
   "never missing", NA, "", "derived", "QC flag", NEVER,
   "NOT AN EXCLUSION (Build 1). 0 for every participant — structurally impossible, since a same-day "
   "dementia would be the nearest diagnosis and the draw could not match as MCI.",
   "no human review pending — the count is zero", "FALSE"),
 D("pre_anchor_dementia_n", "Number of dementia diagnoses BEFORE the anchor", "count", ">=0",
   DXSUM, "DIAGNOSIS", "count of dementia diagnoses strictly before the anchor", "pre-anchor",
   "never missing", NA, "", "derived", "QC flag", NEVER,
   "NOT AN EXCLUSION (generated in Build 1, sampled for review in Build 2). "
   "HUMAN ADJUDICATION IS PENDING.",
   "8 participants have >0 (4 of them are in the primary cohort). A post-anchor dementia in these "
   "people may be RE-DOCUMENTATION of prevalent dementia rather than incident progression.",
   "FALSE"),
 D("qc_dementia_on_or_before_anchor", "FLAG: any dementia diagnosis on or before the anchor",
   "boolean", "TRUE/FALSE", "derived", "(flag)",
   "pre_anchor_dementia_n > 0 OR a same-day dementia", "on/before anchor", "never missing", NA, "",
   "derived", "QC flag", NEVER,
   "NOT AN AUTOMATIC EXCLUSION (Build 1 generated it; Build 2 routed all 8 to human review). "
   "HUMAN ADJUDICATION IS PENDING. Do not exclude these participants without a completed, "
   "recorded team adjudication.",
   "TRUE for 8 participants (4 in the primary cohort, 1 of which carries an event). Excluding all "
   "8 would give a primary cohort of ~397 with ~84 events — a SCENARIO, not a decision.", "FALSE"),
 D("nonpositive_followup_flag", "FLAG: event/censor was not strictly after the anchor", "boolean",
   "FALSE (all rows)", "derived", "(flag)",
   "follow-up time <= 0 -> demoted to no-usable-follow-up", NA, "never missing", NA, "", "derived",
   "QC flag", NEVER,
   "NOT AN EXCLUSION FLAG in practice — FALSE for every participant; no human review pending.",
   "0 occurrences; every follow-up duration is strictly positive", "FALSE"),
]

dd = pd.DataFrame(DD)
dd["data_type"] = dd["output_column"].map(lambda c: str(PRIM[c].dtype))
dd = dd[["output_column", "label", "data_type", "unit", "allowed_values", "source_dataset",
         "source_field", "derivation", "timing_rule", "missing_value_rule", "sentinel_handling",
         "assay_platform", "raw_or_derived", "cohort_role", "transformation_allowed",
         "recommended_downstream_handling", "known_caveat", "authoritative_for_modeling"]]

# ---- ASSERTIONS 8, 9, 10 ----
check("data dictionary covers EVERY column of the primary dataset, in file order",
      list(dd["output_column"]) == PRIM_COLS,
      f"dict={len(dd)}, file={len(PRIM_COLS)}")
check("no primary-dataset column is missing from the dictionary",
      not (set(PRIM_COLS) - set(dd["output_column"])),
      str(set(PRIM_COLS) - set(dd["output_column"])))
check("no dictionary row references a nonexistent column",
      not (set(dd["output_column"]) - set(PRIM_COLS)),
      str(set(dd["output_column"]) - set(PRIM_COLS)))
check("exactly 5 columns are marked authoritative for modeling "
      "(2 outcome + 3 predictors) and they are the contract columns",
      set(dd.loc[dd["authoritative_for_modeling"] == "TRUE", "output_column"])
      == set([DURATION_COL, EVENT_COL] + MODEL1_PREDICTORS),
      sorted(dd.loc[dd['authoritative_for_modeling'] == 'TRUE', 'output_column']))

print(f"\nData dictionary: {dd.shape[0]} rows x {dd.shape[1]} cols")
print(dd["cohort_role"].value_counts().to_string())

  PASS  data dictionary covers EVERY column of the primary dataset, in file order  [dict=78, file=78]
  PASS  no primary-dataset column is missing from the dictionary  [set()]
  PASS  no dictionary row references a nonexistent column  [set()]
  PASS  exactly 5 columns are marked authoritative for modeling (2 outcome + 3 predictors) and they are the contract columns  [['APOE4_COUNT', 'entry_age', 'event_indicator', 'ptau217', 'time_to_event_or_censor_days']]

Data dictionary: 78 rows x 18 cols
cohort_role
provenance                         35
QC flag                            26
SECONDARY PREDICTOR                 5
SECONDARY PREDICTOR (candidate)     4
PRIMARY PREDICTOR                   3
identifier                          2
OUTCOME                             2
OUTCOME (convenience)               1


## 4. Freeze manifest

In [5]:
def git(*a):
    try:
        return subprocess.run(["git", *a], cwd=PROJECT_ROOT, capture_output=True, text=True,
                              check=True).stdout.strip()
    except Exception:
        return None


dd_path = save(dd, f"mci_survival_data_dictionary_{VERSION}.tsv")

SRC_FILES = [Path(s["file"]).name for s in prov1["sources"]]
NOTEBOOKS = ["01c_mci_survival_cohort_freeze.ipynb", "01d_mci_survival_manual_qc_packet.ipynb",
             "01e_mci_survival_feature_availability_audit.ipynb",
             "01f_mci_survival_handoff_package.ipynb"]
SELF_NB = "01f_mci_survival_handoff_package.ipynb"

ALL_OUTPUTS = BUILD1 + BUILD2 + BUILD3 + [f"mci_survival_data_dictionary_{VERSION}.tsv"]

UNRESOLVED = [
 dict(id=1, decision="Prior dementia before the anchor",
      question="Should the 8 participants with a dementia diagnosis BEFORE their MCI anchor be "
               "excluded from the primary analysis?",
      evidence=f"8 in cohort A, {pd_surv} in B ({pd_surv_ev} events), {pd_prim} in C "
               f"({pd_prim_ev} event -> RID {pd_prim_ev_rids}). Exclusion scenario: "
               f"C = {s2_prim} / {s2_prim_ev} events.",
      status="UNRESOLVED", owner="clinical team", blocking_primary_analysis=True,
      artifact="mci_survival_manual_qc_form_v1.tsv (blank; awaiting human review)"),
 dict(id=2, decision="Cognition measured AFTER the anchor",
      question="Is a 'baseline' cognitive score measured up to +88 days after the blood draw "
               "acceptable, or must cognition be restricted to on-or-before the anchor?",
      evidence="post-anchor counts: CDRSB 15, MMSCORE 17, MOCA 24, FAQTOTAL 51. Restricting to "
               "on-or-before removes ALL leakage and costs 0 events for CDRSB and MMSCORE.",
      status="UNRESOLVED", owner="modeling + clinical", blocking_primary_analysis=False,
      artifact="mci_survival_feature_timing_v1.tsv"),
 dict(id=3, decision="Cognitive '-1' not-done sentinel handling",
      question="Should the sentinel rule be extended from {-4,-5} to include -1 for cognitive "
               "fields?",
      evidence="raw MMSE has 14 and raw FAQ has 32 '-1' codes; the pipeline never applies sentinel "
               "cleaning to cognition. NO -1 reached frozen v1, so v1 is unaffected — but the risk "
               "is live for any future cognitive variable.",
      status="UNRESOLVED", owner="data team", blocking_primary_analysis=False,
      artifact="mci_survival_feature_missingness_v1.tsv"),
 dict(id=4, decision="Exact cognition variable selection",
      question="Which cognitive variable (if any) should enter a secondary model?",
      evidence="event retention: MMSCORE 81, FAQTOTAL 81, CDRSB 50, MOCA 41 (of 85). No validated "
               "memory/executive composite exists; CDMEMORY/CDJUDGE are CDR box scores and CDJUDGE "
               "is only an executive PROXY.",
      status="UNRESOLVED", owner="clinical team", blocking_primary_analysis=False,
      artifact="mci_survival_model_complete_case_counts_v1.tsv"),
 dict(id=5, decision="Biomarker transformation / scaling",
      question="Should p-tau217 be log-transformed and/or standardised?",
      evidence="raw pg/mL values are right-skewed. NO transformation has been applied anywhere in "
               "Person 1's outputs — deliberately left to the modeling stage.",
      status="UNRESOLVED", owner="modeling", blocking_primary_analysis=True,
      artifact="mci_survival_primary_cohort_v1.tsv (raw values)"),
 dict(id=6, decision="Prediction horizon",
      question="At what horizon (e.g. 24 months) should risk be reported and calibrated?",
      evidence="events by 12/24/36 months = 15/33/55 of 85 in the primary cohort.",
      status="UNRESOLVED", owner="modeling + clinical", blocking_primary_analysis=True,
      artifact="mci_survival_model_complete_case_counts_v1.tsv"),
 dict(id=7, decision="Penalized versus ordinary Cox",
      question="Should the primary model be an ordinary Cox model or a penalized one?",
      evidence="85 events / 3 predictors ~= 28 events per predictor, which is comfortable for an "
               "ordinary Cox model; penalization is a judgement call, not a data constraint.",
      status="UNRESOLVED", owner="modeling", blocking_primary_analysis=True, artifact=""),
 dict(id=8, decision="Minimum-follow-up sensitivity analyses",
      question="Should any minimum-observed-follow-up restriction be run as a sensitivity analysis?",
      evidence="thresholds 30/90/180/365/730d retain 349/329/314/286/195 survival participants; "
               "NO event is ever removed (removals are exclusively censored). Conditioning on "
               "future follow-up is selection on post-baseline information.",
      status="UNRESOLVED", owner="methodology", blocking_primary_analysis=False,
      artifact="mci_survival_feature_scenario_counts_v1.tsv"),
 dict(id=9, decision="Low / medium / high risk-group definition",
      question="How should predicted risk be discretised into risk groups, if at all?",
      evidence="NOT addressed by Person 1. No cutoff of any kind has been selected.",
      status="UNRESOLVED", owner="clinical + modeling", blocking_primary_analysis=False,
      artifact=""),
]

manifest = {
  "manifest_schema_version": MANIFEST_SCHEMA_VERSION,
  "build": 4, "version": VERSION,
  "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
  "created_utc_note": "THE ONLY non-deterministic field. Excluded from byte-identity comparisons.",
  "project_root": str(PROJECT_ROOT),
  "git": {"commit": git("rev-parse", "HEAD"),
          "branch": git("rev-parse", "--abbrev-ref", "HEAD"),
          "dirty": bool(git("status", "--porcelain"))},
  "environment": {"python": platform.python_version(), "platform": platform.platform(),
                  "pandas": pd.__version__, "numpy": np.__version__,
                  "note": "notebooks 01c-01f require only python + pandas + numpy; "
                          "nbformat/nbconvert/ipykernel are needed to execute them"},
  "self_referential_hashing": {
      "manifest_own_hash": None,
      "policy": "A file cannot contain its own hash. This manifest EXCLUDES its own SHA-256; verify "
                "it externally with `shasum -a 256 mci_survival_freeze_manifest_v1.json` against the "
                "value printed in reports/mci_survival_person1_handoff_v1.md.",
      "notebook_hash_policy": "Notebook FILE hashes change every time nbconvert --inplace rewrites "
                              "stored outputs. We therefore record `source_sha256` = SHA-256 of the "
                              "concatenated CODE CELLS, which is execution-independent and is the "
                              "hash to verify. 01f additionally cannot record its own file hash "
                              "because it writes this manifest during its own execution."},
  "parameters": {"anchor_window_days": ALIGN_TOL_DAYS, "random_seed": RANDOM_SEED,
                 "days_per_month": DAYS_PER_MONTH,
                 "diagnosis_coding": {"1": "CN", "2": "MCI", "3": "Dementia",
                                      "blank": "unknown/other — never an event, never a censor"},
                 "sentinel_rules": {"plasma": "-4/-5 -> NaN; non-positive -> NaN",
                                    "cognition": "NONE APPLIED — see unresolved decision #3",
                                    "known_gap": "a '-1' not-done code exists in raw MMSE/FAQ and is "
                                                 "NOT covered by the -4/-5 rule (none reached v1)"},
                 "imputation": "none", "transformation": "none"},
  "modeling_contract": {
      "authoritative_modeling_file":
          f"outputs/01c_mci_survival_cohort_freeze/{AUTHORITATIVE_FILE}",
      "duration_column": DURATION_COL, "event_column": EVENT_COL,
      "event_coding": {"1": "dementia observed", "0": "right-censored"},
      "model0_predictors": MODEL0_PREDICTORS, "model1_predictors": MODEL1_PREDICTORS,
      "n_participants": N_PRIM, "n_events": EV_PRIM, "n_censored": N_PRIM - EV_PRIM,
      "model0_and_model1_share_identical_participants": True},
  "cohorts": {"anchor": {"n": N_ANCHOR, "events": None},
              "survival_followup": {"n": N_SURV, "events": EV_SURV},
              "primary_complete_case": {"n": N_PRIM, "events": EV_PRIM}},
  "source_data": [
      {"path": f"Data/raw/{f}", "sha256": sha256(RAW_DIR / f),
       "bytes": (RAW_DIR / f).stat().st_size,
       "mtime_utc": datetime.fromtimestamp((RAW_DIR / f).stat().st_mtime,
                                           timezone.utc).isoformat(timespec="seconds")}
      for f in SRC_FILES],
  "notebooks": [
      {"path": f"notebooks/{n}", "source_sha256": nb_source_sha256(NB_DIR / n),
       "file_sha256": (None if n == SELF_NB else sha256(NB_DIR / n)),
       "note": ("SELF: writes this manifest; its own file hash cannot be recorded inside itself — "
                "verify source_sha256" if n == SELF_NB else "")}
      for n in NOTEBOOKS],
  "outputs": [
      {"path": f"outputs/01c_mci_survival_cohort_freeze/{f}", "build": (1 if f in BUILD1 else
                                                                        2 if f in BUILD2 else
                                                                        3 if f in BUILD3 else 4),
       "sha256": sha256(FREEZE_DIR / f),
       "n_rows": (int(pd.read_csv(FREEZE_DIR / f, sep="\t").shape[0]) if f.endswith(".tsv") else None),
       "n_cols": (int(pd.read_csv(FREEZE_DIR / f, sep="\t").shape[1]) if f.endswith(".tsv") else None)}
      for f in ALL_OUTPUTS],
  "reports": ["reports/mci_survival_build0_inventory_v1.md",
              "reports/mci_survival_build1_freeze_report_v1.md",
              "reports/mci_survival_build2_qc_guide_v1.md",
              "reports/mci_survival_build3_feature_audit_v1.md",
              "reports/mci_survival_primary_analysis_spec_v1.md",
              "reports/mci_survival_person1_handoff_v1.md"],
  "qc_status": {"pre_anchor_dementia_flagged": pd_anchor,
                "pre_anchor_dementia_in_survival": pd_surv,
                "pre_anchor_dementia_in_primary": pd_prim,
                "pre_anchor_dementia_primary_events": pd_prim_ev,
                "affected_primary_event_rids": pd_prim_ev_rids,
                "human_adjudication": "PENDING — mci_survival_manual_qc_form_v1.tsv is blank",
                "censor_is_anchor_matching_visit": n_cia,
                "followup_le_30_days": n_fu30},
  "unresolved_decisions": UNRESOLVED,
  "reproduce": ("for nb in 01c_mci_survival_cohort_freeze 01d_mci_survival_manual_qc_packet "
                "01e_mci_survival_feature_availability_audit 01f_mci_survival_handoff_package; do "
                "Data/.venv/bin/python -m nbconvert --to notebook --execute --inplace "
                "--ExecutePreprocessor.timeout=1200 \"notebooks/${nb}.ipynb\" || exit 1; done"),
}

MPATH = FREEZE_DIR / f"mci_survival_freeze_manifest_{VERSION}.json"
MPATH.write_text(json.dumps(manifest, indent=2, default=str))
print(f"  saved -> {MPATH.relative_to(PROJECT_ROOT)}")
print(f"\nManifest covers: {len(manifest['source_data'])} source files, "
      f"{len(manifest['notebooks'])} notebooks, {len(manifest['outputs'])} outputs, "
      f"{len(UNRESOLVED)} unresolved decisions")

  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_data_dictionary_v1.tsv   (78 x 18)


  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_freeze_manifest_v1.json

Manifest covers: 8 source files, 4 notebooks, 16 outputs, 9 unresolved decisions


## 5. Validation

In [6]:
# ---- ASSERTION 11, 12 ----
missing_files, bad_hash = [], []
for rec in manifest["outputs"]:
    p = PROJECT_ROOT / rec["path"]
    if not p.exists():
        missing_files.append(rec["path"])
    elif sha256(p) != rec["sha256"]:
        bad_hash.append(rec["path"])
for rec in manifest["source_data"]:
    p = PROJECT_ROOT / rec["path"]
    if not p.exists():
        missing_files.append(rec["path"])
    elif sha256(p) != rec["sha256"]:
        bad_hash.append(rec["path"])
for rec in manifest["notebooks"]:
    if not (PROJECT_ROOT / rec["path"]).exists():
        missing_files.append(rec["path"])
for r in manifest["reports"]:
    if not (PROJECT_ROOT / r).exists():
        print(f"  NOTE: report not yet written (expected on first run): {r}")

check("every file recorded in the manifest EXISTS", not missing_files, str(missing_files))
check("every hash recorded in the manifest MATCHES the current file", not bad_hash, str(bad_hash))

# ---- ASSERTION 13: Builds 1-3 byte-identical ----
HASHES_AFTER = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
drift = [f for f in PROTECTED if HASHES_BEFORE[f] != HASHES_AFTER[f]]
check("ALL Build 1-3 artifacts are BYTE-IDENTICAL before and after Build 4",
      not drift, f"drifted: {drift}" if drift else f"{len(PROTECTED)} artifacts unchanged")

# ---- ASSERTION 14: no frozen inclusion flag changed ----
_re = pd.read_csv(FREEZE_DIR / AUTHORITATIVE_FILE, sep="\t")
check("no frozen inclusion flag was changed",
      bool(_re["primary_eligible"].astype(bool).all()) and len(_re) == N_PRIM
      and int((_re[EVENT_COL] == 1).sum()) == EV_PRIM)

# ---- ASSERTION 15: no scientific decision silently resolved ----
check("all 9 scientific decisions remain UNRESOLVED (none silently answered)",
      len(UNRESOLVED) == 9 and all(d["status"] == "UNRESOLVED" for d in UNRESOLVED),
      f"{len(UNRESOLVED)} decisions, all UNRESOLVED")
check("no v2 / revised cohort file was created",
      not any(p.name.endswith(("_v2.tsv", "_v2.json")) for p in FREEZE_DIR.iterdir()))

# ---- ASSERTION 16 ----
check("the manifest identifies EXACTLY ONE authoritative modeling file, and it exists",
      isinstance(manifest["modeling_contract"]["authoritative_modeling_file"], str)
      and (PROJECT_ROOT / manifest["modeling_contract"]["authoritative_modeling_file"]).exists(),
      manifest["modeling_contract"]["authoritative_modeling_file"])

print(f"\nMANIFEST SHA-256 (record this — it is not inside the file):\n  {sha256(MPATH)}")
print(f"AUTHORITATIVE MODELING INPUT SHA-256:\n  {sha256(FREEZE_DIR / AUTHORITATIVE_FILE)}")

  PASS  every file recorded in the manifest EXISTS  [[]]
  PASS  every hash recorded in the manifest MATCHES the current file  [[]]
  PASS  ALL Build 1-3 artifacts are BYTE-IDENTICAL before and after Build 4  [15 artifacts unchanged]
  PASS  no frozen inclusion flag was changed
  PASS  all 9 scientific decisions remain UNRESOLVED (none silently answered)  [9 decisions, all UNRESOLVED]


  PASS  no v2 / revised cohort file was created
  PASS  the manifest identifies EXACTLY ONE authoritative modeling file, and it exists  [outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_v1.tsv]

MANIFEST SHA-256 (record this — it is not inside the file):
  55f6d945ee340e4f20402c71bb56eaca1d3fc813e2cda7a79436df48ea1526d4
AUTHORITATIVE MODELING INPUT SHA-256:
  a15e2cb8606f676b3bcfc4d23de72c752779836ce9406b04b01c84394a76e8ad


## 6. Build 4 summary

In [7]:
print("=" * 92)
print("BUILD 4 — DATA DICTIONARY, PROVENANCE & MODELING HANDOFF (v1)")
print("=" * 92)
print(f"Assertions : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print()
print("MODELING CONTRACT (verified against the file, not assumed):")
print(f"  file      : outputs/01c_mci_survival_cohort_freeze/{AUTHORITATIVE_FILE}")
print(f"  N         : {N_PRIM}   events: {EV_PRIM}   censored: {N_PRIM - EV_PRIM}")
print(f"  duration  : {DURATION_COL}")
print(f"  event     : {EVENT_COL}  (1 = dementia, 0 = censored)")
print(f"  Model 0   : {' + '.join(MODEL0_PREDICTORS)}")
print(f"  Model 1   : {' + '.join(MODEL1_PREDICTORS)}")
print(f"  -> both models use the SAME {N_PRIM} participants and {EV_PRIM} events")
print()
print(f"Data dictionary : {len(dd)} rows (one per column), {int((dd['authoritative_for_modeling']=='TRUE').sum())} authoritative")
print(f"Manifest        : {len(manifest['outputs'])} outputs, {len(manifest['source_data'])} sources, "
      f"{len(manifest['notebooks'])} notebooks hashed")
print()
print("PENDING HUMAN ADJUDICATION:")
print(f"  {pd_anchor} pre-anchor-dementia participants remain in frozen v1 "
      f"({pd_prim} in C, {pd_prim_ev} event -> RID {pd_prim_ev_rids})")
print(f"  exclusion SCENARIO would give C = {s2_prim} / {s2_prim_ev} events (NOT applied)")
print()
print(f"{len(UNRESOLVED)} scientific decisions recorded as UNRESOLVED.")
print("Builds 1-3 byte-identical. Frozen v1 unchanged.")
print("=" * 92)

BUILD 4 — DATA DICTIONARY, PROVENANCE & MODELING HANDOFF (v1)
Assertions : 25 passed / 0 failed  (of 25)

MODELING CONTRACT (verified against the file, not assumed):
  file      : outputs/01c_mci_survival_cohort_freeze/mci_survival_primary_cohort_v1.tsv
  N         : 401   events: 85   censored: 316
  duration  : time_to_event_or_censor_days
  event     : event_indicator  (1 = dementia, 0 = censored)
  Model 0   : entry_age + APOE4_COUNT
  Model 1   : entry_age + APOE4_COUNT + ptau217
  -> both models use the SAME 401 participants and 85 events

Data dictionary : 78 rows (one per column), 5 authoritative
Manifest        : 16 outputs, 8 sources, 4 notebooks hashed

PENDING HUMAN ADJUDICATION:
  8 pre-anchor-dementia participants remain in frozen v1 (4 in C, 1 event -> RID [467])
  exclusion SCENARIO would give C = 397 / 84 events (NOT applied)

9 scientific decisions recorded as UNRESOLVED.
Builds 1-3 byte-identical. Frozen v1 unchanged.
